 # Utility Function Version 2: Now with Total Utility

Basic idea is to take version 1, clean it up, and then figure out a way to calculate total utility for each voter in one period

## 1.0 Basic Code

### 1.1 Import Packages

In [3]:
import numpy as np
import scipy
import random
import pandas as pd

seed_for_prng = 78557
prng = np.random.default_rng(seed_for_prng)

print("NumPy version:", np.__version__)
print("SciPy version:", scipy.__version__)
print("Pandas version:", pd.__version__)

from scipy.optimize import linprog
from scipy.optimize import minimize



NumPy version: 1.23.5
SciPy version: 1.15.3
Pandas version: 2.3.1


### 1.2 Define Variables

In [5]:
#How many voters?
num_voters = 4
print("Number of voters: ", num_voters)

#How many policies?
num_policies = 3
print("Number of policies: ", num_policies)

#What is our elasticity of substitution?
rho = 0.5
print("Rho (Elasticity of Substitution)= ", rho)

Number of voters:  4
Number of policies:  3
Rho (Elasticity of Substitution)=  0.5


# 2.0 CES Function

In [1]:
def ces_utility(y, alpha, rho):
    y = np.array(y)
    
    # Disallow negative quantities
    if np.any(y < 0):
        return -np.inf
    
    # Guard against undefined behavior when rho < 0 and y == 0
    if rho < 0 and np.any(y == 0):
        return -np.inf
    
    if np.isclose(rho, 0.0):
        # Cobb-Douglas limit
        return np.prod(y ** alpha)
    else:
        return (np.sum(alpha * y**rho))**(1/rho)

def maximize_ces_n_goods(alpha, rho, budget):
    alpha = np.array(alpha)
    n = len(alpha)
    
    # Normalization check
    if not np.isclose(np.sum(alpha), 1.0):
        raise ValueError("Weights alpha must sum to 1.")
    
    # All prices = 1 by assumption
    prices = np.ones(n)
    
    def objective(y):
        return -ces_utility(y, alpha, rho)
    
    # Budget constraint (equality, since CES utility is monotonic)
    constraints = [{'type': 'eq', 'fun': lambda y: budget - np.sum(prices * y)}]
    
    # Non-negativity bounds
    bounds = [(0, None) for _ in range(n)]
    
    # Improved starting point based on alpha (p_i = 1)
    y0 = budget * alpha

    result = minimize(objective, y0, bounds=bounds, constraints=constraints)

    if result.success:
        y_opt = result.x
        utility = ces_utility(y_opt, alpha, rho)
        return {
            'quantities': y_opt,
            'utility': utility
        }
    else:
        raise RuntimeError("Optimization failed: " + result.message)


# 3.0 Expertise, Preference and Delegation

### 3.1 Expertise and Preference Generation

In [ ]:
#Voter expertise by policy (measured on a scale of 1-5)
expertise_matrix = pd.DataFrame(
    prng.integers(1, 6, size = (num_voters, num_policies)),
    columns = [f"policy_{i}" for i in range(num_policies)],
    index = [f"voter_{i}" for i in range(num_voters)])
print("Voter expertise by policy:")
print(expertise_matrix)
print()
#This generates a dataframe with num_voters rows and num_policies columns, each with a prng between 1-5.
#This is meant to indicate how much expertise each voter has on each policy

simple_voter_expertise = np.zeros(num_voters) #simplifies our expertise matrix, which used to be by policy, into a single value
for i in range(num_voters):
    simple_voter_expertise[i] = np.mean(expertise_matrix.iloc[i])
print("Simplified voter expertise:")
print(simple_voter_expertise)
print()
#We could use a prng to assign single expertise values and not bother at all with per-policy expertise, since how PPV is designed, you either delegate to a voter i or you delegate your vote to a policy, but cannot delegate to voter i for a specific policy; voter i also can exercise liquid democracy.
#So in this regard per-policy expertise doesn't really matter, but I am keeping the code here in case I need it in the future.


print()

#Voter preference by policy
preference_matrix = pd.DataFrame(
    prng.integers(1, 6, size = (num_voters, num_policies)),
    columns = [f"policy_{i}" for i in range(num_policies)],
    index = [f"voter_{i}" for i in range(num_voters)])
print("Voter preference by policy:")
print(preference_matrix)
print()
#Similar to expertise, each voter has a preference for each policy, their policy position measured on a numerical scale of 1-5 scale (for example, 1=highly disapprove 5=highly approve, or it could be the other way around).
#As the code is written right now, the 1-5 values do not need to be ordinal on a single spectrum, as we have yet to implement whether or not a policy gets implemented (1-2=vote no, 3=vote neutral, 4-5=vote yes). However, it does seem odd that someone would delegate their vote to an issue only to vote neutral, so maybe when we do do yes/no voting it shouldbe on a scale of 1-4?
#However, if this is changed to 1-4, then we have a problem where since the values of expertise and preference are ordinal, and right now they are compared the same (i.e. vote delegation = expertise + preference), expertise would have a higher ceiling than preference (max of 5 vs 4 respectively). This could be solved by using negative numbers. 

def matrix_value_call(voter, policy, called_matrix): #calls specific values within any matrix. Note: this is zero indexed
    return called_matrix.iloc[voter, policy]

Voter expertise by policy:
         policy_0  policy_1  policy_2
voter_0         1         4         2
voter_1         3         3         3
voter_2         4         5         3
voter_3         4         2         5

Simplified voter expertise:
[2.33333333 3.         4.         3.66666667]


Voter preference by policy:
         policy_0  policy_1  policy_2
voter_0         3         3         3
voter_1         2         5         1
voter_2         1         4         2
voter_3         2         4         5


### 3.2 Preference Alignment

In [11]:
#This code generates a three-dimensional matrix to store policy alignment between different voters for different policies, with the axes voter i, voter j, and policy p respectively.
def complete_preference_calculator ():
    complete_preference_matrix = np.zeros((num_voters, num_voters, num_policies))
    for p in range(num_policies):
        for i in range(num_voters):
            for j in range(num_voters):
                complete_preference_matrix[i, j, p] = 5 - np.abs(matrix_value_call(i, p, preference_matrix) - (matrix_value_call(j,p,preference_matrix))) #for a specific policy, if the alignment is exact, then value = 5. If it differs by 1 (i.e. 2 vs 3) then the alignment is 4.If they differ completely (1 vs 5) then the alignment is 1.
    return complete_preference_matrix

complete_preference_alignment = complete_preference_calculator()

def simple_preference_calculator (big_pref_matrix): #takes the complete preference alignment matrix, which measures i x j preference alignment for every policy, and averages the values to generate a single number for i  j prefernce alignment
    simple_preference_matrix = np.zeros((num_voters, num_voters))
    for i in range(num_voters):
        for j in range(num_voters):
            simple_preference_matrix[i, j] = np.mean(big_pref_matrix[i, j, :])
    return simple_preference_matrix

simple_preference_alignment = simple_preference_calculator (complete_preference_alignment)

print("Simple preference alignment:")
print(simple_preference_alignment)

Simple preference alignment:
[[5.         3.33333333 3.66666667 3.66666667]
 [3.33333333 5.         4.         3.33333333]
 [3.66666667 4.         5.         3.66666667]
 [3.66666667 3.33333333 3.66666667 5.        ]]


## 4.0 Delegation

### 4.1 Voter j Delegates to Voter i

In [12]:
#for voter j, delegate to all voter i. When this is completed, run for all voter js.

def j_delegates_to_i (j, rho): #If we really wanted to be picky, then we could make every j voter have a different rho; if not we could just define it globally.
    #the CES function has inputs of alpha, rho, and budget. Rho = whatever, budget = 100. Thus alpha must be defined

    alpha_list = np.zeros(num_voters) #Since there are as many alpha values as goods/i voters (delegatees), we start by generating a blank list of n length
    for i in range(len(alpha_list)):
        alpha_list[i] = (simple_preference_alignment[i, j] + simple_voter_expertise[i]) #calculates the alpha of each voter i, or in other words, the attractiveness of delegating to them, which is a sum of preference alignment and expertise.
    alpha_list = alpha_list/np.sum(alpha_list) #Since our CES funcion requires that all alpha sum to 1, this normalizes our values
    return maximize_ces_n_goods (alpha_list, rho, 100) #maximizes CES function with alpha values (Preference alignment + expertise), rho (some value), and budget=100

### 4.2 Voter j to Voter i Delegation Stacking

In [ ]:
def voter_to_voter_delegations (rho):
    delegation_utility_list = np.zeros(num_voters) #empty utility list
    n_x_n_matrix = np.array(j_delegates_to_i (0, rho)['quantities']).reshape(1, -1) #because we are using numpy array matrix, we have to first calculate the first j and then add onto that
    delegation_utility_list [0] = j_delegates_to_i (0, rho)['utility'] #Here we calculate the first j's delegations and utility

    for j in range(1, num_voters):
        new_row = np.array(j_delegates_to_i(j, rho)['quantities']).reshape(1, -1)
        n_x_n_matrix = np.vstack((n_x_n_matrix, new_row))
        delegation_utility_list[j] = j_delegates_to_i(j, rho)['utility']
    return {
            'delegation_matrix': n_x_n_matrix,
            'utility': delegation_utility_list
        }
print("Voter j to Voter i Delegation Matrix:")
print(voter_to_voter_delegations(rho)['delegation_matrix'])
print()
print("Voter j to Voter i Delegation Utility:")
print(voter_to_voter_delegations(rho)['utility'])
print()

Voter j to Voter i Delegation Matrix:
[[26.13378088 19.39993188 28.33250797 26.13377927]
 [15.35738681 30.60527879 30.60522639 23.43210801]
 [16.38585895 22.24199713 36.84964191 24.52250202]
 [17.13745087 19.10485734 27.99011514 35.76757666]]

Voter j to Voter i Delegation Utility:
[25.12160769 25.44618712 25.54233964 25.55435353]



Matrix is then transposed as the final matrix has delegators as columns and delegatees as rows

In [15]:
voter_to_voter_matrix = np.transpose(voter_to_voter_delegations(rho)['delegation_matrix']) #This calls the function from section 4 and assigns it to a variable. Nothing special.
#However, we may need to globally define rho in the future, when running the entire code all at once.

voter_to_voter_utility = voter_to_voter_delegations(rho)['utility']

### 4.3 Voter j Delegates to Policy p

In [16]:
#We go through the same methodology, first having each voter delegate to policies, and then afterwards stacking this all up into a matrix

def j_delegates_to_p (j, rho): #Similar framework for j_delegates_to_i, but with policy factors
    alpha_list = preference_matrix.iloc[j] #Our alphas for voter j are just their policy preferences for all policy p, which is already contained in voter j's row in our preference_matrix
    alpha_list = alpha_list/np.sum(alpha_list) #Since our CES funcion requires that all alpha sum to 1, this normalizes our values
    return maximize_ces_n_goods (alpha_list, rho, 100) #maximizes CES function with alpha values, rho (some value), and budget=100

### 4.4 Voter j to Policy p Delegation Stacking

In [21]:
def voter_to_policy_delegations (rho):
    delegation_utility_list = np.zeros(num_voters) #empty utility list
    n_x_p_matrix = np.array(j_delegates_to_p (0, rho)['quantities']).reshape(1, -1) #First j's policy delegations
    delegation_utility_list [0] = j_delegates_to_p (0, rho)['utility'] #Here we calculate the first j's delegations and utility

    for j in range(1, num_voters):
        new_row = np.array(j_delegates_to_p(j, rho)['quantities']).reshape(1, -1)
        n_x_p_matrix = np.vstack((n_x_p_matrix, new_row))
        delegation_utility_list[j] = j_delegates_to_p(j, rho)['utility']
    return {
            'delegation_matrix': n_x_p_matrix,
            'utility': delegation_utility_list
        }

print("Voter j to Policy p Delegation Matrix:")
print(voter_to_policy_delegations(rho)['delegation_matrix'])
print()
print("Voter j to Policy p Delegation Utility:")
print(voter_to_policy_delegations(rho)['utility'])
print()

Voter j to Policy p Delegation Matrix:
[[33.33333333 33.33333333 33.33333333]
 [13.33048664 83.33705554  3.33245781]
 [ 4.76187113 76.19042593 19.04770294]
 [ 8.88906712 35.54643903 55.56449384]]

Voter j to Policy p Delegation Utility:
[33.33333333 46.87499988 42.85714286 37.19008229]



Matrix is then transposed as the final matrix has delegators as columns and delegatees as rows

In [26]:
voter_to_policy_matrix = np.transpose(voter_to_policy_delegations(rho)['delegation_matrix'])

voter_to_policy_utility = voter_to_policy_delegations(rho)['utility']

## 5.0 Creating the Full Delegation Matrix

### 5.1 Creating Policy to Voter Matrix (Zero) and Policy to Policy Matrix (Identity)

In [31]:
policy_to_voter_matrix = np.zeros((num_voters, num_policies))
policy_to_policy_matrix = np.identity((num_policies)) * 100

### 5.2 Compiling the Full Delegation Matrix (Raw)

In [32]:
full_delegation_matrix = np.block([
    [voter_to_voter_matrix, policy_to_voter_matrix],
    [voter_to_policy_matrix, policy_to_policy_matrix] 
])

def normalize_full_delegation_matrix():
    for j in range(num_voters):
        for p in range(num_voters, num_voters+num_policies):
            full_delegation_matrix[p, j] = full_delegation_matrix[p, j]/100 * full_delegation_matrix[j,j]
        full_delegation_matrix[j, j] = 0

normalize_full_delegation_matrix()

full_delegation_matrix = full_delegation_matrix/100 #this is mainly just to normalize the whole thing so that everything is in percentages

np.set_printoptions(precision=5, suppress=True) #limits decimals for neatness


print("Full Delegation Matrix:")
print(full_delegation_matrix)

Full Delegation Matrix:
[[0.      0.15357 0.16386 0.17137 0.      0.      0.     ]
 [0.194   0.      0.22242 0.19105 0.      0.      0.     ]
 [0.28333 0.30605 0.      0.2799  0.      0.      0.     ]
 [0.26134 0.23432 0.24523 0.      0.      0.      0.     ]
 [0.08711 0.0408  0.01755 0.03179 1.      0.      0.     ]
 [0.08711 0.25506 0.28076 0.12714 0.      1.      0.     ]
 [0.08711 0.0102  0.07019 0.19874 0.      0.      1.     ]]
